## Running MS2LDA

In [1]:
%matplotlib agg
import sys, os
sys.path.append('..') 
import tomotopy as tp
from matchms.importing import load_from_mgf
from MS2LDA.Preprocessing.load_and_clean import clean_spectra
import MS2LDA

In [ ]:
import tomotopy as tp
ms2lda = tp.LDAModel.load("CaseStudy_siderophores_300_230226/ms2lda.bin")
len(ms2lda.docs) #we have 2714 preprocessed spectra

8262

In [ ]:
top_motifs = []

#This function helps to obtain the top motifs per spectrum

for i in range(len(ms2lda.docs)):
    topics = ms2lda.docs[i].get_topics()
    doc_motifs = []  
    for motif in topics:
        if motif[1] > 0.09:
            doc_motifs.append(motif)
    top_motifs.append(doc_motifs) 


In [4]:
len(top_motifs)

8262

In [5]:
top_motifs[205] #Here are for example the top motifs for the precursor 617 reported in your paper

[(66, 0.9851394891738892)]

In [6]:
top_motifs[1306] #you can see these two precursors share motif 66

[(66, 0.997282087802887)]

In [7]:
spectra_list = []

In [8]:
from MS2LDA.utils import retrieve_spec4doc
import pickle
from pathlib import Path

In [9]:
pkl_path = Path("/Users/rtlortega/Documents/PhD/WP1/WP1/Code/MS2LDA/notebooks/Paper_results/CaseStudy_siderophores_300_230226_4/doc2spec_map.pkl")

with pkl_path.open("rb") as f:
    doc2spec_map = pickle.load(f)


In [10]:
for i in range(len(ms2lda.docs)):
    spectra_list.append(retrieve_spec4doc(doc2spec_map, ms2lda, i))

In [11]:
spectra_list[1306].metadata #trace back amphibactin F

{'scans': '1921',
 'charge': 1,
 'collision_energy': '0.0',
 'retention_time': 699.599,
 'ms_level': '2',
 'precursor_mz': 876.5269,
 'ionmode': 'positive',
 'retention_index': None,
 'id': 'spec_1306'}

In [12]:
scans_list = []
precursor_list = []

for i in spectra_list:
    scan = i.get("scans")
    prec = i.get("precursor_mz")
    scans_list.append(scan)
    precursor_list.append(prec)

In [13]:
import pandas as pd

df = pd.DataFrame({
    'scans': scans_list,
    'motifs': top_motifs,
    'precursor_mz' : precursor_list
})

print(df.head())


  scans                       motifs  precursor_mz
0    99  [(163, 0.9943950176239014)]      376.2599
1     2  [(176, 0.9961933493614197)]      311.0812
2    35   [(56, 0.8742567300796509)]      338.3421
3     5   [(205, 0.996896505355835)]      186.9564
4    82  [(163, 0.9835754632949829)]      393.2865


In [14]:
df.columns

Index(['scans', 'motifs', 'precursor_mz'], dtype='object')

In [15]:
df_long = (
    df[["scans", "precursor_mz", "motifs"]]
    .explode("motifs")
    .dropna(subset=["motifs"])
    .assign(
        motif_id=lambda x: x["motifs"].apply(lambda t: t[0]),
        prob=lambda x: x["motifs"].apply(lambda t: t[1]),
    )
    .drop(columns="motifs")
    .rename(columns={"scans": "scan"})
    .reset_index(drop=True)
)

df_long

,scan,precursor_mz,motif_id,prob
0,99,376.2599,163,0.994395
1,2,311.0812,176,0.996193
2,35,338.3421,56,0.874257
3,5,186.9564,205,0.996897
4,82,393.2865,163,0.983575
...,...,...,...,...
18455,15363,325.2370,118,0.489206
18456,15363,325.2370,47,0.240997
18457,15364,294.2065,146,0.690898
18458,15364,294.2065,277,0.232434


In [16]:
df_long.to_csv('Siderophores_Neha_M2M.tsv', index=False, sep='\t')

In [19]:
motif_66_df = pd.read_csv("Motif_66_filtered_prob0.09.csv")
motif_531_df = pd.read_csv("MS2LDA_MOTIFDB-576d04bd-view_all_motifs-main.tsv", sep="\t")
motif_66_df.head()

,scan,precursor_mz,motif_id,prob
0,363,589.3809,66,0.987806
1,370,615.3965,66,0.966516
2,380,617.4121,66,0.985139
3,430,643.4274,66,0.975108
4,448,603.3965,66,0.985707


In [23]:
import numpy as np

In [24]:
m1 = motif_531_df["precursor.mass"].to_numpy(float)   # df1
m2 = motif_66_df["precursor_mz"].to_numpy(float)      # df2

ppm = 3
diff = np.abs(m1[:, None] - m2[None, :])
tol  = (m2[None, :] * ppm * 1e-6)
match = diff <= tol

# Correct axis interpretation:
motif_66_overlapped  = int(match.any(axis=0).sum())   # df2 rows matched
motif_531_overlapped = int(match.any(axis=1).sum())   # df1 rows matched

print("Rows in df2 motif 66 matched by df1 motif_531:", motif_66_overlapped, "out of", len(motif_66_df))
print("Rows in df1 motif_531 matched by df2 motif 66:", motif_531_overlapped, "out of", len(motif_531_df))

Rows in df2 motif 66 matched by df1 motif_531: 32 out of 68
Rows in df1 motif_531 matched by df2 motif 66: 31 out of 136


---